In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import openpyxl
import re
import os
from variableUtils import *
import variableUtils
from Utils import *
from ClassUtils import *
from pprint import pprint
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from reportlab.lib.pagesizes import letter, landscape, A4, A3
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, PageBreak, Paragraph, Spacer, Image
from reportlab.lib import colors
from reportlab.platypus import Paragraph, Spacer, KeepTogether, KeepInFrame
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
import io
from math import sqrt, pi, exp
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import FormulaRule
import PIL
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(sns.__version__)
# get today's date
today = datetime.now().strftime('%d-%m-%Y')
print(f"Today's date: {today}")
plt.ioff()

(841.68, 1190.8799999999999)
0.13.2
Today's date: 03-11-2025


In [2]:
workbookpath = '2025\BOH3+DDS4\Session 8\\2025 BOH3 and DDS4 CAF_Merged.xlsx'
# workbookpath = '2025\BOH3+DDS4\\2025 BOH3 and DDS4.xlsx'
def getStudentList(listpath, cohort='BOH3'):
    listdf = pd.read_excel(listpath, keep_default_na=False, na_values=[''])
    if cohort is not None:
        listdf = listdf[listdf['Cohort'] == cohort]
    validIds = listdf['Student ID'].to_list()
    # convert to int
    validIds = [int(id) for id in validIds]
    print(f"Number of valid student IDs in {cohort}: {len(validIds)}")
    return validIds
validIds = getStudentList('2025\RE_ Student List.xlsx', cohort=None)
folder, file, ext = getFolderandFileName(workbookpath)
df = pd.read_excel(workbookpath, keep_default_na=False, na_values=[''])  # This will keep 'NA' text as it is
# turn evey column to lowercase
df.columns = df.columns.str.lower()
# strip all columns
df.columns = df.columns.str.strip()
# remove non-ascii characters
df.columns = df.columns.str.replace(r'[^\x00-\x7F]+', '', regex=True)
df.loc[0] = df.loc[0].str.replace(r'[^\x00-\x7F]+', '', regex=True)
supervisorcol = [col for col in df.columns if 'evaluation#2' in col.lower()]
studentcol = [col for col in df.columns if 'evaluation#1' in col.lower()]
# display(df[supervisorcol[3]].value_counts(dropna=False))
fullnamedict = {col: df.loc[0, col] for col in df.columns}
row0 = df.loc[0]
# row0 = row0.str.replace(r'[^\x00-\x7F]+', '', regex=True)
df.drop(0, inplace=True)
# df['finished'] = df['finished'].astype(bool)
colNPts = 'No. of pts'
colId = 'student id'
colOprStuFeedback = 'student reflection'
colCI = 'Clinical Incident'
colCIExp = 'CI Explanation'
colRotation = 'Rotation'
colResponseId = 'ResponseId'
colFinished = 'Finished'
colName = 'student name'
colNameF = 'family name'
colNameG = 'given name'
df[colName] = df[colNameF] + ' ' + df[colNameG]
df = df[(df['finished'] == True) | (df['finished'] == 'TRUE') | (df['finished'] == 'true') | (df['finished'] == 'True')]
# change Additional clinic to 5 in rotation column
df['rotation'] = df['rotation'].replace({'Additional clinic': 5, 'Additional Clinic': 5, 'additional clinic': 5})
df['rotation'] = pd.to_numeric(df['rotation'], errors='coerce')
df['rotation'] = df['rotation'].astype('Int64')
df = df[df['rotation'] != 8]
colAge = [col for col in df.columns if '#2_1_1' in col]
colPtInfo = [col for col in df.columns if re.match(r'pt\.\s*\d+', col)]
def getAgeGroup(age):
    if age <= 6:
        return '0-6'
    elif age <= 17:
        return '7-17'
    elif age >= 18 and age<140:
        return '18+'
    else:
        return np.nan
    
for col in colAge:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # replace all values that are not numbers with np.nan
    # df[col] = df[col].apply(lambda x: x if isinstance(x, (int, float)) else np.nan)
colAgeNew = []
for col in colAge:
    df[f'{col} group'] = df[col].apply(getAgeGroup)
    colAgeNew.append(f'{col} group')

# display(df[df[colId.lower()]==1452727][colAge])
colCode = [col for col in df.columns if 'code' in col.lower()]
colCodeNew = []
for col in colCode:
    df[col] = df[col].astype(str)
    df[f'{col} list'] = df[col].apply(lambda x: [f'0{item}' if item.isdigit() and len(item) == 2 else item for item in re.split(r',\s*', x)])  # Handles optional spaces after commas and adds 0 before two-digit items
    colCodeNew.append(f'{col} list')
# display(df)

for col in colCodeNew:
    df[col] = df[col].apply(cleanEntry)

# print(colAge)
# display(df.loc[0, colAge])
colCohort = 'cohort'

# display(dfboh3.head())


colPriority = [col for col in df.columns if 'priority' in col.lower()]
colDate = 'date'
# filter out valid student IDs
df = df[df[colId].isin(validIds)]
originalDates = df[colDate].copy()

# Try parsing with coercion
df[colDate] = pd.to_datetime(df[colDate], errors='ignore', format='mixed')

# Find the actual problematic original dates
invalidDateValues = originalDates[df[colDate].isna()].unique()

print(f"Invalid date values: {invalidDateValues}")
df[colDate] = df[colDate].dt.strftime('%d %b %Y')
# display(df[colDate].value_counts(dropna=False))
# patient info columns are with Pt.\d+ in the column name

# print(colPtInfo)
importantCols =  ['SO Feedback',	'SO Edu Feedback',	'SO Edu Name', 'Obs Feedback',	'Obs Edu Feedback', 'Obs Edu Name']
columnList = [
    "Student Reflection",
    "Edu Feedback",
    "Entrustment",
    "Clinical Incident",
    "CI Explanation",
    "Edu Name"
]
importantCols.extend(columnList)
print(fullnamedict)
print(df[colFinished.lower()].value_counts(dropna=False))
df.set_index(colResponseId.lower(), inplace=True)

print(folder)
# dfboh3[dfboh3[colId]==1462377]

Number of valid student IDs in None: 619
Invalid date values: []
{'responseid': 'Response ID', 'startdate': 'Start Date', 'enddate': 'End Date', 'status': 'Response Type', 'ipaddress': 'IP Address', 'progress': 'Progress', 'duration (in seconds)': 'Duration (in seconds)', 'finished': 'Finished', 'recordeddate': 'Recorded Date', 'recipientlastname': 'Recipient Last Name', 'recipientfirstname': 'Recipient First Name', 'recipientemail': 'Recipient Email', 'externalreference': 'External Data Reference', 'locationlatitude': 'Location Latitude', 'locationlongitude': 'Location Longitude', 'distributionchannel': 'Distribution Channel', 'userlanguage': 'User Language', 'given name': 'Student Given Name', 'family name': 'Student Family Name', 'student id': 'Student ID', 'email': 'Student Email Address', 'date': 'Date', 'cohort': 'Cohort', 'rotation': 'Rotation Number', 'clinic': 'Clinic', 'clinic-other': 'Clinic- Other', 'role': 'Student Role\n\n\n\nStudents who do not see a patient during a cli

In [3]:
# create id to name mapping
id_to_name = df.set_index(colId)[colName].to_dict()
print(id_to_name)

{1082098: 'Nguyen Jennifer', 984694: 'Chen Xiyang', 1377061: 'DEVGUN MahirKumar', 1461909: 'Jahan Agrin', 1205045: 'Brar Arashdeep', 1280725: 'Cornelio Rohan', 1221841: 'Dhaliwal Ashley', 913592: 'Chen Austina', 1472277: 'Fakhrualdin Melak', 1472810: 'Wong Ruth', 1461325: 'Ho Jason', 1452767: 'Vuong Amy', 1461861: 'Wang Oliver', 1452755: 'Malellari Rayyan', 1353765: 'Lee JungMin', 1462810: "Koo MooK'PohPaw", 1461394: 'Tran QuynhThiDiem', 1460391: 'Nguyen Celina', 987912: 'Lau TszChing', 1472645: 'Feng Alicia', 1452672: 'Unal Esmanur', 1129001: 'Naylor Kaelan', 1452727: 'Khan Anshrah', 995150: 'Luo Louisa', 916569: 'Owuama Henry', 1218058: 'Mistry Parth', 974490: 'Ing Damie', 1213090: 'Sidhu Gurjot', 1452784: 'Ktaifan Ranim', 1222331: 'Lynn Travis', 1469829: 'Zheng Wenhui', 1035907: 'Koh BuoYu', 1222329: 'Hammad Ahmed', 985101: 'Zhan Zeyu', 608989: 'Atwal Nimret', 1080271: 'Fok Edwin', 734343: 'Son Yukyung', 1443777: 'Zaki Natalie', 1460683: 'Chen Sitong', 1419850: 'Khatibi Zahra', 1473

### Analysis on item codes

In [4]:
colAllCodes = 'all codes'
# combine all code columns into a single list
df[colAllCodes] = df[colCodeNew].sum(axis=1)
display(df[[colId, colAllCodes]].head())
dfboh3 = df[df[colCohort]=='BOH3']
dfdds4 = df[df[colCohort]=='DDS4']

,student id,all codes
responseid,,
R_4DdAEeA2cSleg70,1082098,"[012, 141, 114, 121]"
R_4mOFtm7pXUGin6s,984694,"[012, 115, 121, 141, 221, 222]"
R_407PoEF7hyCo2UN,1377061,[]
R_4pzMt1X2AZIKKRT,1461909,"[012, 114, 141, 121, 111, 141]"
R_4zGFfc2qC9m0GMj,1205045,"[013, 061, 533]"


In [5]:
def getcodesbyidandrotation(df: pd.DataFrame, colId: str= colId.lower(), 
                            colRotation: str = colRotation.lower(), colAllCodes: str = colAllCodes) -> pd.DataFrame:
    """ 
    Get student codes by student ID and rotation.
    returns: DataFrame
    """
    
    print(f"getting student codes by {colId} and {colRotation}")
    studentcodes= df.groupby([colId, colRotation])[colAllCodes].apply(sum).reset_index()
    return studentcodes

def getcodesbyid(df: pd.DataFrame, colId: str = colId.lower(),
                  colAllCodes: str = colAllCodes) -> pd.DataFrame:
    """
    Get student codes by student ID.
    returns: DataFrame
    """
    print(f"getting student codes by {colId}")
    studentcodes= df.groupby([colId])[colAllCodes].apply(sum).reset_index()
    display(studentcodes.head())
    return studentcodes

def sanitizeLists(x) -> list:
    if isinstance(x, list): 
        return x
    if pd.isna(x): 
        return []
    return [x]

def buildStudentCodeTables(df, idCol=colId.lower(), codesCol=colAllCodes) -> pd.DataFrame:
    """
    Components & functionality:
    - sanitizeLists: ensures each row has a list (handles NaN/strings)
    - longDf: one row per (studentId, code)
    - longCounts: counts of each code per studentId
    - pivotDf: pivot table with studentId rows, code columns, counts as values    
    """
    codesbyid = getcodesbyid(df, colId=colId.lower(), colAllCodes=colAllCodes)
    tmp = codesbyid.copy()
    tmp[codesCol] = tmp[codesCol].apply(sanitizeLists)

    longDf = tmp.explode(codesCol).dropna(subset=[codesCol])
    longDf = longDf.rename(columns={idCol: 'studentId'})
    longDf['code'] = longDf[codesCol].astype(str).str.strip()

    longCounts = (
        longDf.groupby(['studentId', 'code'])
              .size()
              .reset_index(name='count')
              .sort_values(['studentId', 'code'])
    )

    pivotDf = (
        longCounts.pivot(index='studentId', columns='code', values='count')
                 .fillna(0)
                 .astype(int)
                 .sort_index(axis=1)
    )

    return longCounts, pivotDf

def computeKde1d(values, gridPoints=256):
    """Gaussian KDE (Silverman bandwidth) without seaborn/scipy."""
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    n = x.size
    if n < 2 or np.std(x) == 0:
        return None, None
    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmin == xmax:
        xmin, xmax = xmin - 0.5, xmax + 0.5
    grid = np.linspace(xmin, xmax, gridPoints)
    h = 1.06 * np.std(x, ddof=1) * (n ** (-1/5))
    if not np.isfinite(h) or h <= 0:
        return None, None
    # Vectorized Gaussian sum
    z = (grid[:, None] - x[None, :]) / h
    dens = (np.exp(-0.5 * z * z).sum(axis=1) / (n * h * sqrt(2 * pi)))
    return grid, dens

def plotHistogramWithKde(ax, series, bins=30, title=""):
    """Histogram (density) + KDE line for one code."""
    data = pd.to_numeric(series, errors='coerce').dropna().values
    if data.size == 0:
        ax.text(0.5, 0.5, "No data", ha='center', va='center')
        ax.set_title(title)
        ax.set_xlabel("Count")
        ax.set_ylabel("Density")
        return
    ax.hist(data, bins=bins, density=True, alpha=0.6)
    grid, dens = computeKde1d(data)
    if grid is not None:
        ax.plot(grid, dens, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Count")
    ax.set_ylabel("Density")

def saveCodeHistogramsToPdf(pivotDf, pdfPath, bins=30,
                            pageSize=A4, topMargin=36, bottomMargin=36,
                            leftMargin=36, rightMargin=36, dpi=160):
    """
    Components & functionality:
    - computeKde1d: fast 1D KDE (Gaussian, Silverman bandwidth)
    - plotHistogramWithKde: draws histogram+density for one code
    - saveCodeHistogramsToPdf: loops columns, renders PNGs, embeds in PDF
    """
    doc = SimpleDocTemplate(pdfPath, pagesize=pageSize,
                            topMargin=topMargin, bottomMargin=bottomMargin,
                            leftMargin=leftMargin, rightMargin=rightMargin)
    styles = getSampleStyleSheet()
    story = []

    # Render one chart per code (column)
    for code in pivotDf.columns:
        # skip if total is less than 1
        if pivotDf[code].sum() < 10:
            continue
        fig, ax = plt.subplots(figsize=(8, 4.5))
        plotHistogramWithKde(ax, pivotDf[code], bins=bins, title=f"Code {code} — Count Distribution")
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', dpi=dpi)
        plt.close(fig)
        buf.seek(0)

        story.append(Paragraph(f"Code {code}", styles['Heading2']))
        story.append(Image(buf, width=480, height=270))  # auto-kept aspect via bbox crop
        story.append(Spacer(1, 12))
        story.append(PageBreak())

    if story and isinstance(story[-1], PageBreak):
        story.pop()  # remove last extra page break

    doc.build(story)
    return pdfPath

def flagStudentsBehind(pivotDf, methods=('percentile','zscore','ratio'),
                       percentile=0.30, zThresh=-1.0, ratio=0.5):
    """
    Components & functionality:
    - counts: numeric matrix of code counts (rows=studentId, cols=code)
    - statsDf: per-code summary (mean, median, std, q10)
    - masks: boolean flags per method:
        * percentile: count <= per-code q{percentile}
        * zscore: z = (x-mean)/std, flag if z < zThresh
        * ratio: count < median*ratio
    - behindMask: OR of selected masks
    - behindLong: table of (studentId, code) that are behind
    """
    counts = pivotDf.copy().apply(pd.to_numeric, errors='coerce').fillna(0)

    codeMean = counts.mean(axis=0)
    codeMean = codeMean.reindex(counts.columns)  # Ensure same order as columns
    codeStd  = counts.std(axis=0).replace(0, np.nan)
    codeStd = codeStd.reindex(counts.columns)  # Ensure same order as columns
    codeMed  = counts.median(axis=0)
    codeMed = codeMed.reindex(counts.columns)  # Ensure same order as columns
    codeQ    = counts.quantile(percentile, axis=0)
    codeQ = codeQ.reindex(counts.columns)    # Ensure same order as columns
    print(codeQ)
    statsDf = pd.DataFrame({
        'mean': codeMean, 'std': codeStd, 'median': codeMed, f'q{int(percentile*100)}': codeQ
    }).sort_index()
    # display(statsDf.head())
    masks = {}

    # if 'percentile' in methods:
    #     masks['percentile'] = counts.le(codeQ)  # <= qX
    # Z-score rule with mean<std fallback
    if 'zscore' in methods:
        z = (counts - codeMean) / codeStd
        zMask = z.lt(zThresh)
        meanLtStdCols = ((codeMean < codeStd)&(codeMean>2)).fillna(False)
        if meanLtStdCols.any():
            meanMask = counts.lt(codeMean, axis='columns')
            zMask.loc[:, meanLtStdCols] = meanMask.loc[:, meanLtStdCols]
        masks['zscore'] = zMask

    if 'ratio' in methods:
        masks['ratio'] = counts.lt(codeMed * ratio)
    # display(masks['percentile'].head())
    behindMask = None
    for m in masks.values():
        behindMask = m if behindMask is None else (behindMask | m)

    behindLong = (
        behindMask.stack().rename('isBehind').reset_index()
        .rename(columns={'level_0': 'studentId', 'level_1': 'code'})
        .query('isBehind').drop(columns='isBehind')
        .sort_values(['studentId','code'])
    )
    display(behindLong.head())
    behindByStudent = (
    behindLong.groupby('studentId', sort=True)['code']
      .agg(lambda x: list(dict.fromkeys(map(str, x))))
      .reset_index(name='codes')
        )
    behindByStudent['codes'] = behindByStudent['codes'].apply(lambda x: ", ".join(x))
    return {'statsDf': statsDf, 'masks': masks, 'behindMask': behindMask, 'behindLong': behindByStudent}

def styleBehind(pivotDf, behindMask):
    """Returns a Styler highlighting behind cells."""
    return pivotDf.style.apply(lambda _:
        behindMask.replace({True: 'background-color: #ffe6e6', False: ''}),
        axis=None
    )

keepCodes = ['414']
def performCohortCodeAnalysis(df, cohort='All', removeUnflagged = True):
    codecounts, codecountspivot = buildStudentCodeTables(df)
    display(codecounts.head())
    display(codecountspivot.head())
    os.makedirs(f"{folder}/stats", exist_ok=True)
    codecountspivot.to_excel(f"{folder}/stats/{cohort}_codecountspivot.xlsx")
    pdfPath = f"{folder}/stats/{cohort}_code_histograms.pdf"
    saveCodeHistogramsToPdf(codecountspivot, pdfPath)
    leftbehind = flagStudentsBehind(codecountspivot)
    if removeUnflagged:
        colFlags  = leftbehind['behindMask'].any(axis=0)                 # True if any student is flagged for that code
        keptCols  = list(colFlags[colFlags].index) + keepCodes
        droppedCols = [c for c in codecountspivot.columns if c not in keptCols]
        pivotFiltered = codecountspivot.loc[:, keptCols] if keptCols else codecountspivot.iloc[:, 0:0]
        maskFiltered  = leftbehind['behindMask'].loc[:, keptCols]        if keptCols else leftbehind['behindMask'].iloc[:, 0:0]
        statsFiltered = leftbehind['statsDf']#.loc[keptCols]              if keptCols else leftbehind['statsDf'].iloc[0:0]
    
    else:
        pivotFiltered, maskFiltered, statsFiltered = codecountspivot, leftbehind['behindMask'], leftbehind['statsDf']
        keptCols, droppedCols = list(codecountspivot.columns), []

    pivotFiltered[colName] = pivotFiltered.index.map(id_to_name)
    maskFiltered[colName] = False
    pivotFiltered = pivotFiltered[[colName]+[col for col in pivotFiltered.columns if col != colName]]
    maskFiltered = maskFiltered[[colName]+[col for col in maskFiltered.columns if col != colName]]
    styler = styleBehind(pivotFiltered, maskFiltered)
    # add student name
    leftbehind['behindLong'][colName] = leftbehind['behindLong']['studentId'].map(id_to_name)
    leftbehind['behindLong'] = leftbehind['behindLong'][[colName, 'codes']]
    filePath = f'{folder}/stats/{cohort}_behind.xlsx'
    with pd.ExcelWriter(filePath, engine='openpyxl') as writer:
        styler.to_excel(writer, sheet_name='pivot')                 # highlighted + filtered
        statsFiltered.to_excel(writer, sheet_name='codeStats')      # filtered stats
        leftbehind['behindLong'].to_excel(writer, sheet_name='behindList', index=False)
        if droppedCols:
            pd.DataFrame({'code': droppedCols}).to_excel(writer, sheet_name='droppedCodes', index=False)

        ws = writer.sheets['pivot']
        ws.freeze_panes = 'B2'
        ws.auto_filter.ref = ws.dimensions

performCohortCodeAnalysis(dfboh3, cohort='BOH3')
performCohortCodeAnalysis(dfdds4, cohort='DDS4')
# codecountspivot.to_excel(f"{folder}/stats/codecountspivot.xlsx")
def plotcodesfromlist(allCodes:list, title:str, figsize:tuple = (figSize[0], figSize[1])):
    
    """
    Plot all codes in subplots from a list of codes.
    returns: Figure
    """
    # remove if more than 5 characters
    code_counts = pd.Series(allCodes).value_counts(ascending=False)
    # remove counts less than 2
    code_counts = code_counts[code_counts >= 2]
    if len(code_counts) == 0:
        code_counts = pd.Series(allCodes).value_counts(ascending=False)

    # Divide into subplots with no more than 20 bars each
    num_codes = len(code_counts)
    maxbars = 20 if num_codes <=80 else 25 
    num_subplots = int(np.ceil(num_codes / maxbars))
    fig, axes = plt.subplots(num_subplots, 1, figsize=figsize)
    if num_subplots == 1:
        axes = [axes]  # Ensure axes is iterable for a single subplot

    for i, ax in enumerate(axes):
        start_idx = i * maxbars
        end_idx = start_idx + maxbars
        subset = code_counts[start_idx:end_idx]
        sns.barplot(x=subset.index, y=subset.values, ax=ax)
        # ax.text(0.9, 0.9, f'Total: {len(codes)}', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        for j, v in enumerate(subset.values):
            if max(subset.values) > 4:
                ax.text(j, v + 0.5, str(v), color='black', ha='center', fontsize=8)
            else:
                ax.text(j, v/2, str(v), color='black', ha='center', fontsize=8)
            
        ax.set_title(f'{title} (Part {i + 1})')
        if len(axes) == 1:
            ax.set_title(f'{title}')
        ax.set_xlabel('Items')
        ax.set_ylabel('Frequency')
        ax.set_ylim(0, max(subset.values)*1.1)  # Set y-axis limit to accommodate text
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

    plt.tight_layout()
    return fig

getting student codes by student id


,student id,all codes
0,1270550,"[011, 121, 111, 012, 022, 061, 141, 114, 011, ..."
1,1280725,"[221, 221, 115, 141, 011, 221, 141, 131, 111, ..."
2,1337652,"[011, 022, 024, 531, 531, 531, 114, 222, 141, ..."
3,1353765,"[221, 141, 935, 012, 111, 141, 114, 123, 013, ..."
4,1377061,"[013, 113, 115, 141, 013, 072, 022, 114, 221, ..."


,studentId,code,count
0,1270550,011,80
1,1270550,012,18
2,1270550,013,14
3,1270550,015,1
4,1270550,018,5


code,001,002,010,011,012,013,014,015,016,017,018,019,021,022,023,024,025,026,027,028,036,037,038,039,041,046,047,048,051,052,055,061,064,065,067,069,070,071,072,073,074,075,079,084,091,092,099,100,111,112,113,114,115,116,121,122,123,131,134,141,142,144,151,155,161,162,165,171,211,212,213,221,222,231,250,280,311,316,331,351,411,414,431,513,514,521,522,523,524,525,529,531,532,533,534,535,537,553,562,568,571,572,574,577,578,586,587,618,627,655,735,741,753,754,766,776,779,789,799,811,857,863,872,911,916,919,924,927,935,937,941,961,965,970,981,986,990,991,LA
studentId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1270550,0,0,0,80,18,14,0,1,0,0,5,6,0,60,0,30,0,0,0,0,0,7,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,0,2,54,7,0,33,0,6,35,0,48,0,0,0,0,30,5,0,0,0,0,0,15,39,0,0,0,3,0,0,0,0,0,0,0,0,5,2,0,0,0,0,17,1,0,0,0,0,0,0,0,0,0,0,0,0,1,6,0,0,0,0,0,0,0,0,0,0,0,13,0,0,0,0,0,23,0,0,0,1,0,0,0,0,0,0,0,0,0,7
1280725,0,0,0,30,7,32,0,0,0,0,0,3,0,55,0,7,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,6,25,18,0,12,0,1,8,0,47,0,0,0,0,3,0,0,1,0,0,0,17,32,0,0,0,1,0,0,0,0,0,0,1,0,3,3,2,2,0,0,6,16,1,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0,1,0,41
1337652,0,0,0,54,16,14,1,0,1,0,0,6,0,68,0,12,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,7,0,0,54,13,0,27,0,18,3,0,24,0,0,0,0,76,0,0,0,1,0,0,13,7,0,0,0,6,0,0,0,0,0,0,0,0,9,0,3,2,0,0,12,6,1,1,0,0,0,0,0,0,1,0,1,0,0,7,0,0,0,0,0,0,0,0,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1353765,0,0,0,46,14,13,0,0,0,0,0,2,0,58,1,14,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,4,0,4,50,9,0,12,0,6,0,0,21,0,0,0,0,13,0,0,0,0,0,0,14,8,0,0,0,5,0,0,0,0,0,0,0,0,1,1,0,0,0,0,13,4,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,8,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,20
1377061,0,0,0,53,7,34,1,0,0,0,0,5,0,72,0,20,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,3,0,0,1,0,0,0,0,0,0,6,0,3,32,10,0,22,0,5,33,0,65,1,1,0,0,22,0,0,0,0,0,0,10,6,0,0,0,3,0,0,0,0,0,0,0,0,1,1,0,0,0,0,3,11,1,1,0,0,0,0,0,0,0,0,0,0,0,7,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,2,0,0,0,0,11


code
001     0.0
002     0.0
010     0.0
011    55.8
012     9.0
013    18.0
014     0.0
015     0.0
016     0.0
017     0.0
018     0.0
019     5.0
021     0.0
022    69.9
023     0.0
024    14.9
025     0.0
026     0.0
027     0.0
028     0.0
036     0.0
037     3.9
038     0.0
039     0.0
041     0.0
046     0.0
047     0.0
048     0.0
051     0.0
052     0.0
055     0.0
061     3.0
064     0.0
065     0.0
067     0.0
069     0.0
070     0.0
071     0.0
072     1.0
073     0.0
074     0.0
075     0.0
079     0.0
084     0.0
091     0.0
092     0.0
099     0.0
100     0.0
111     4.0
112     0.0
113     2.0
114    46.9
115     9.0
116     0.0
121    28.0
122     0.0
123     5.0
131     6.9
134     0.0
141    50.0
142     0.0
144     0.0
151     0.0
155     0.0
161    23.9
162     0.0
165     0.0
171     0.0
211     0.0
212     0.0
213     0.0
221    13.0
222    19.8
231     0.0
250     0.0
280     0.0
311     3.0
316     0.0
331     0.0
351     0.0
411     0.0
414     0.0
431     0.0

...

,student id,all codes
0,608989,"[011, 531, 061, 141, LA, 061, 022, 013, 131, 3..."
1,693184,"[011, 221, 114, 222, 222, 011, 221, 011, 011, ..."
2,734343,"[013, 022, 024, 419, 019, 013, 022, 024, 311, ..."
3,759735,"[531, 114, 521, 764, 013, 011, 022, 061, 024, ..."
4,832527,"[013, 013, 013, 013, 741, 013, 022, 013, 022, ..."


,studentId,code,count
0,608989,003,1
1,608989,011,20
2,608989,012,6
3,608989,013,133
4,608989,014,11


code,000,001,002,003,011,012,013,014,015,016,017,018,019,021,022,023,024,025,026,027,028,031,032,033,034,036,037,038,039,041,042,043,044,046,047,051,052,055,061,062,063,064,065,071,072,073,074,075,077,085,091,103,109,111,112,113,114,115,116,117,119,121,122,123,131,132,134,141,142,144,151,161,162,165,171,211,213,221,222,223,225,230,231,238,250,252,297,300,307,311,312,314,316,317,321,322,324,326,331,341,342,361,384,386,392,398,399,411,412,414,415,416,417,418,419,423,424,436,445,451,455,472,491,511,512,513,514,521,522,523,524,525,526,527,529,531,532,533,534,535,537,542,543,544,545,553,554,555,561,571,572,573,574,575,577,578,579,586,587,596,597,613,615,618,625,627,628,631,632,642,643,649,651,652,653,655,656,658,659,684,711,712,714,716,721,722,723,724,727,728,731,732,733,736,738,741,742,743,744,746,748,751,752,753,754,755,761,762,763,764,765,766,767,768,769,772,773,774,776,779,786,788,790,799,811,872,873,874,875,877,881,911,912,915,919,925,926,927,931,935,937,952,961,965,966,968,975,978,981,982,986,987,990,991,LA
studentId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
608989,0,0,0,1,20,6,133,11,1,0,0,1,20,0,107,0,21,0,0,0,0,0,0,0,0,0,22,0,0,0,0,0,0,0,0,0,0,1,44,0,0,0,0,4,1,0,0,0,0,0,0,0,0,1,0,12,36,4,0,0,0,27,0,5,1,0,0,20,0,0,0,5,0,0,7,0,4,7,4,0,0,0,0,0,0,0,0,0,0,71,0,2,0,0,0,1,2,0,1,0,0,0,0,0,0,0,0,0,0,0,2,5,1,2,4,0,0,0,0,0,4,0,0,0,0,0,0,4,4,2,2,0,0,0,0,32,26,7,4,1,0,0,0,0,0,0,0,0,0,0,8,0,1,0,7,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,1,2,0,1,0,0,0,0,0,0,0,3,0,0,0,0,3,0,0,0,0,0,0,0,1,0,0,0,0,0,3,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16
693184,0,0,0,0,43,6,81,0,0,0,0,0,8,0,102,0,44,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,1,19,0,0,0,0,2,1,0,0,0,0,0,0,0,0,3,0,4,36,7,0,0,0,21,0,9,2,0,0,32,0,0,0,23,6,0,1,0,0,14,55,0,0,0,0,0,0,0,0,0,0,36,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,7,0,0,0,0,0,2,0,0,0,0,0,0,12,6,3,4,3,0,0,0,31,24,4,4,3,0,0,0,0,0,0,0,0,0,0,8,0,0,0,3,3,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,1,0,0,0,3,0,2,3,0,0,2,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,3,0,0,0,0,0,0,0,0,0,0,1,0,3,0,0
734343,0,0,0,0,30,5,116,3,0,1,0,0,22,0,100,0,39,0,0,0,0,0,0,0,0,0,10,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,4,0,0,0,0,0,0,0,0,1,0,13,36,5,4,0,0,24,0,16,4,0,0,40,2,0,0,6,0,4,2,0,0,8,39,0,0,0,0,0,0,0,0,0,0,59,0,0,12,0,0,1,4,0,2,0,0,0,0,0,2,0,0,0,0,0,1,0,1,0,6,0,0,0,0,0,3,0,0,0,0,0,0,19,12,6,3,0,0,0,0,19,15,9,1,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,3,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,0,0,0,0,0,0,0,0,0,0,1,2,4,5,0,0,1,0,0,0,0,17,0,0,0,0,4,0,0,0,0,0,0,0,1,0,0,0,0,0,8,0,7,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
759735,0,0,0,0,28,4,78,5,0,0,0,1,7,0,87,0,25,0,0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0,1,0,0,0,64,0,0,0,0,4,4,1,0,0,0,0,0,0,0,4,0,13,44,3,0,0,0,32,0,1,3,0,0,29,0,0,1,14,0,0,1,0,1,14,21,0,0,0,1,0,0,0,0,0,0,23,0,1,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,4,1,1,0,3,0,0,0,0,0,0,0,0,0,0,0,0,14,10,5,2,1,0,0,0,34,29,7,2,1,0,0,0,0,0,0,0,0,0,0,9,0,0,0,9,0,0,1,3,1,0,0,2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,4,1,2,0,0,0,1,0,0,0,0,3,0,0,0,0,8,0,0,0,0,0,0,0,2,0,0,0,0,0,2,0,5,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0
832527,0,0,0,0,37,14,91,1,0,0,0,0,5,0,105,1,28,0,0,0,0,0,0,0,0,0,12,0,0,0,0,0,0,0,1,0,0,0,42,0,0,0,0,5,4,1,0,0,0,0,0,0,0,3,0,4,32,5,0,0,0,32,0,1,1,0,0,33,0,0,0,15,0,2,1,0,0,12,11,0,0,0,0,0,0,0,0,0,0,28,0,0,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,4,0,0,0,0,0,0,0,0,0,0,0,0,13,15,6,4,1,0,0,0,26,28,11,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,8,0,0,3,0,0,0,0,1,0,2,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,3,2,1,0,9,0,0,0,0,0,0,0,0,0,0,0,0,1,2,1,0,0,0,0,1,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,4,0,0,0,1,0,0,0,0,0,0,0,0,5,0,0


code
000      0.0
001      0.0
002      0.0
003      0.0
011     27.0
012      6.0
013     91.0
014      3.0
015      0.0
016      0.0
017      0.0
018      0.0
019     11.0
021      0.0
022    105.0
023      0.0
024     27.0
025      0.0
026      0.0
027      0.0
028      0.0
031      0.0
032      0.0
033      0.0
034      0.0
036      0.0
037      8.0
038      0.0
039      0.0
041      0.0
042      0.0
043      0.0
044      0.0
046      0.0
047      0.0
051      0.0
052      0.0
055      0.0
061     27.0
062      0.0
063      0.0
064      0.0
065      0.0
071      1.0
072      3.0
073      0.0
074      0.0
075      0.0
077      0.0
085      0.0
091      0.0
103      0.0
109      0.0
111      1.0
112      0.0
113      4.0
114     33.0
115      3.0
116      0.0
117      0.0
119      0.0
121     22.0
122      0.0
123      2.0
131      1.0
132      0.0
134      0.0
141     22.0
142      0.0
144      0.0
151      0.0
161      2.0
162      0.0
165      0.0
171      2.0
211      0.0
213    

,studentId,code
4,608989,011
44,608989,072
63,608989,123
64,608989,131
71,608989,161


In [ ]:
def getNPatients(text):
    # print(text)
    if pd.isna(text):
        return 0
    # select the number of patients from the text, select the highest number Patient 1, Patient 2, Patient 3 etc.
    numbers = re.findall(r'\d+', text)
    if len(numbers) == 0 and 'fta' in text.lower():
        return 'Patient FTA'
    # get the highest number
    if len(numbers):
        return max([int(num) for num in numbers])
    else:
        return 0
    
def createGeneralReport(df, cohort):
    idCounts = df[colId].value_counts()
    ids = idCounts.index
    elements = []
    patientCountDf = pd.DataFrame(columns=['Student ID', 'Student Name', '# Patients', 'Patient Ages',
                                           '#18+', '#7-17', '#0-6', 'Clinic'])
    evalDf = pd.DataFrame(columns=['Student ID', 'Student Name', '# Yes', '# No', '# N/A', " No%"])
    
    for id in ids:
        studentDf = df[df[colId] == id]
        studentName = studentDf['given name'].values[0] + ' ' + studentDf['family name'].values[0]
        studentDf['Patient Count'] = studentDf[colNPts.lower()].apply(getNPatients)
        # get fta count
        studentDf['Patient Count'] = studentDf['Patient Count'].replace('Patient FTA', 0).astype(int)
        patientCount = studentDf['Patient Count'].sum()
        ageList = []
        for col in colAge:
            studentDf[col] = studentDf[col].astype(float)
            ageList.extend(studentDf[col].dropna().to_list())
        ageList.sort()
        n18plus = len([age for age in ageList if age >= 18])
        n7to17 = len([age for age in ageList if age >= 7 and age < 18])
        n0to6 = len([age for age in ageList if age >= 0 and age < 7])
        clinic = studentDf['clinic'].values[0]
        patientCountDf = pd.concat([patientCountDf, pd.DataFrame({'Student ID': [id], 'Student Name': [studentName], '# Patients': [patientCount],
             "Patient Ages": [ageList], '#18+': [n18plus], '#7-17': [n7to17], '#0-6': [n0to6], 'Clinic': [clinic]})])
    
        # get the evaluation counts for supervisorcol
        nYes = 0
        nNo = 0
        nNA = 0
        
        for col in supervisorcol:
            nYes += studentDf[col].str.contains('Yes').sum()
            nNo += studentDf[col].str.contains('No').sum()
            nNA += studentDf[col].str.contains('NA').sum()
        nopercent = nNo / (nYes + nNo + nNA + 0.001) * 100
        evalDf = pd.concat([evalDf, pd.DataFrame({'Student ID': [id], 'Student Name': [studentName],
                                                  '# Yes': [nYes], '# No': [nNo], '# N/A': [nNA], ' No%': [nopercent]})])
    
    evalDf.sort_values(by=[' No%', '# Yes'], ascending=[True, False], inplace=True)
    
    # Create a stacked plot of evaluation counts with student id as x axis
    fig, ax = plt.subplots(figsize=(figSize[0], figSize[1]))
    evalDf.set_index('Student ID')[['# Yes', '# No', '# N/A']].iloc[::-1].plot(kind='barh', stacked=True, ax=ax, width=0.8)
    ax.set_xlabel('Count')
    ax.set_ylabel('Student ID')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    plt.subplots_adjust(hspace=0.5)
    plt.tight_layout()
    img = addPlotImage(fig, 0.9)
    
    elements.append(Paragraph(f'Performance Overview for {cohort}', subsubheadingStyle))
    elements.append(Spacer(1, 12))
    elements.append(img)
    elements.append(Spacer(1, 6))
    elements.append(Paragraph("The students at the bottom are with highest % of No Responses", tableTextStyle))
    
    # Create a table for patientCountDf with bar graphs in the Patient Ages column
    table_data = [['Student ID', 'Student Name', '# Patients', 'Patient Ages', 'Age count', 'Clinic']]
    patientCountDf.sort_values(by='# Patients', ascending=False, inplace=True)
    

    unique_clinics = patientCountDf['Clinic'].unique()
    print(unique_clinics)
    clinic_colors = {clinic: color for clinic, color in zip(unique_clinics, sns.color_palette("husl", len(unique_clinics)))}
    
    for i, row in patientCountDf.iterrows():
        # Create a tiny bar graph for the Patient Ages column
        fig, ax = plt.subplots(figsize=(2, 0.5))
        sns.histplot(row['Patient Ages'], bins=[i for i in range(0, 100, 6)], ax=ax)
        ax.yaxis.set_visible(False)
        ax.xaxis.set_visible(False)
        ax.axvline(x=6.5, color='black', linestyle='dotted')
        ax.axvline(x=18, color='black', linestyle='dotted')
        plt.tight_layout()
        imgdata = io.BytesIO()
        fig.savefig(imgdata, format='png', bbox_inches='tight')
        plt.close(fig)
        imgdata.seek(0)
        img = Image(imgdata, width=4 * inch, height=1 * inch)
        agedict = {'18+': row['#18+'], '7-17': row['#7-17'], '0-6': row['#0-6']}
        agetext = '<br/>'.join([f'{key}: {value}' for key, value in agedict.items()])
        clinic_ = row['Clinic'] if not pd.isna(row['Clinic']) else ''
        table_data.append([row['Student ID'], Paragraph(row['Student Name'], tableTextStyle),
                           row['# Patients'], img, Paragraph(agetext, tableTextStyle), Paragraph(clinic_, tableTextStyle)])
        
    table = Table(table_data, colWidths=[1.2 * inch, 1.5 * inch, 1 * inch, 4 * inch, 1.2 * inch, 1.2 * inch])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 0.2 * inch),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 12),
        ('BOTTOMPADDING', (3, 1), (3, -1), 0),  
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
    ]))
    elements.append(PageBreak())
    elements.append(Paragraph(f'Patient Count and Age Distribution for {cohort}', subsubheadingStyle))
    elements.append(Spacer(1, 12))
    text = f"""
    The table shows ages of patients with patient count and age group distribution.The dotted lines in the graphs indicate the age groups 0-6 and 7-17.
    """
    text2 = f"""Average Patient count is {round(patientCountDf['# Patients'].mean(), -1)} per student. """
    text3 = f"""The highest number of patients seen by a student is {patientCountDf['# Patients'].max()}."""
    text4 = f"""The lowest number of patients seen by a student is {patientCountDf['# Patients'].min()}.""" 
    text5 = "Data is not very accurate due to missing fields in the forms but should be close."
    elements.append(Paragraph(text, tableTextStyle))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(text2, tableTextStyle))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(text3, tableTextStyle))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(text4, tableTextStyle))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(text5, tableTextStyle))
    elements.append(Spacer(1, 12))

    elements.append(table)
    
    # doc.build(elements)
    # display(patientCountDf)
    # display(evalDf)
    return elements
# savefolder = f'{folder}'
# createGeneralReport(dfboh3, 'BOH3', savefolder)